In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

print(spark.version)

3.5.0


In [4]:
# create our first dataframe

data = [
    ("George", 99),
    ("Alex", 199),
    ("Victor", 98)
]

df = spark.createDataFrame(data, ["Name", "Age"])

df

DataFrame[Name: string, Age: bigint]

In [5]:
df.show()

+------+---+
|  Name|Age|
+------+---+
|George| 99|
|  Alex|199|
|Victor| 98|
+------+---+



In [6]:
df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Age: long (nullable = true)



- Transformations
- Actions

In [8]:
df.select("Age").show()

+---+
|Age|
+---+
| 99|
|199|
| 98|
+---+



In [9]:
df.select("Age", "Name").show()

+---+------+
|Age|  Name|
+---+------+
| 99|George|
|199|  Alex|
| 98|Victor|
+---+------+



In [11]:
df.select(df.Age, df.Name).show()

+---+------+
|Age|  Name|
+---+------+
| 99|George|
|199|  Alex|
| 98|Victor|
+---+------+



In [12]:
df.columns

['Name', 'Age']

In [13]:
df.filter(df.Age < 100).show()

+------+---+
|  Name|Age|
+------+---+
|George| 99|
|Victor| 98|
+------+---+



In [14]:
df.filter("Age < 100").show()

+------+---+
|  Name|Age|
+------+---+
|George| 99|
|Victor| 98|
+------+---+



In [23]:
from pyspark.sql.functions import col, avg, min, max

In [16]:
df.filter(col("Age") > 100).show()

+----+---+
|Name|Age|
+----+---+
|Alex|199|
+----+---+



In [19]:
# adding a new column

df_new_age = df.withColumn("New_Age", col("Age") + 10)
df_new_age.show()

+------+---+-------+
|  Name|Age|New_Age|
+------+---+-------+
|George| 99|    109|
|  Alex|199|    209|
|Victor| 98|    108|
+------+---+-------+



In [20]:
(df.withColumn("New_Age", col("Age") + 10)).show()

+------+---+-------+
|  Name|Age|New_Age|
+------+---+-------+
|George| 99|    109|
|  Alex|199|    209|
|Victor| 98|    108|
+------+---+-------+



In [22]:
df.select(avg("Age")).show()

+--------+
|avg(Age)|
+--------+
|   132.0|
+--------+



In [24]:
df.select(min("Age")).show()

+--------+
|min(Age)|
+--------+
|      98|
+--------+



In [25]:
df.select(max("Age")).show()

+--------+
|max(Age)|
+--------+
|     199|
+--------+



## Reading CSV File

In [28]:
%pip install pandas

import pandas as pd

pd.read_csv("students.csv").head()

Note: you may need to restart the kernel to use updated packages.


,Name,Age,Score
0,John,22,80
1,Mary,25,90
2,Peter,30,70


In [43]:
df = spark.read.csv("students.csv", header=True, inferSchema=True)

df.show()

+-----+---+-----+
| Name|Age|Score|
+-----+---+-----+
| John| 22|   80|
| Mary| 25|   90|
|Peter| 30|   70|
+-----+---+-----+



In [31]:
df.printSchema()

root
 |-- Name: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Score: integer (nullable = true)



In [32]:
df.select(avg("Age")).show()

+------------------+
|          avg(Age)|
+------------------+
|25.666666666666668|
+------------------+



In [37]:
df.filter(col("Age") >= 25).show()

+-----+---+-----+
| Name|Age|Score|
+-----+---+-----+
| Mary| 25|   90|
|Peter| 30|   70|
+-----+---+-----+



In [39]:
# ascending
df.orderBy("Score").show()

+-----+---+-----+
| Name|Age|Score|
+-----+---+-----+
|Peter| 30|   70|
| John| 22|   80|
| Mary| 25|   90|
+-----+---+-----+



In [40]:
# ascending
df.orderBy(df.Score.asc()).show()

+-----+---+-----+
| Name|Age|Score|
+-----+---+-----+
|Peter| 30|   70|
| John| 22|   80|
| Mary| 25|   90|
+-----+---+-----+



In [41]:
# descending
df.orderBy(df.Score.desc()).show()

+-----+---+-----+
| Name|Age|Score|
+-----+---+-----+
| Mary| 25|   90|
| John| 22|   80|
|Peter| 30|   70|
+-----+---+-----+



In [42]:
df.filter(df.Name == "John").show()

+----+---+-----+
|Name|Age|Score|
+----+---+-----+
|John| 22|   80|
+----+---+-----+



In [66]:
df = spark.read.csv("FinalBV_V8.csv", header=True, inferSchema=True)

df.show()

+--------------------+--------------------+--------------+-----+-----+----------+-------------+-----------------+--------------+-------------+--------------------+--------------------+------------------+--------------------+-----------+-----------------+--------------------+----------+---------------+--------------------+----+----+
|        Company Name|      OpenCorporates|          City|State|  ZIP|PPP Amount|Jobs Reported|jurisdiction code|Company Number|       Status|  Incorporation Date|        Company Type|      Jurisdiction|  Registered Address|Agent_first|agent_middle_name|agent_middle_initial|agent_last|      Full Name|       Agent Address|_c20|_c21|
+--------------------+--------------------+--------------+-----+-----+----------+-------------+-----------------+--------------+-------------+--------------------+--------------------+------------------+--------------------+-----------+-----------------+--------------------+----------+---------------+--------------------+----+----

In [73]:
df.select("City", "State", "Status", "agent_last").limit(15).show()

+-----------+-----+-------------+----------+
|       City|State|       Status|agent_last|
+-----------+-----+-------------+----------+
| Haleyville|   AL|       Exists|     HICKS|
|       NULL| NULL|         NULL|      NULL|
|Saint Louis|   MO|       Active|Beckemeier|
|     Mobile|   AL|       Exists|   SHERMAN|
|       NULL| NULL|         NULL|      NULL|
|   Meredith|   NH|Good Standing|     Brown|
|Osage Beach|   MO|       Active|     SUPER|
|       NULL| NULL|         NULL|      NULL|
|   Columbia|   MO|       Active|    Martin|
|      Foley|   AL|       Exists|    ALLDAY|
|       NULL| NULL|         NULL|      NULL|
|Springfield|   MO|       Active|      Hite|
|       NULL| NULL|         NULL|      NULL|
| Birmingham|   AL|       Exists|      HALL|
|       NULL| NULL|         NULL|      NULL|
+-----------+-----+-------------+----------+



In [59]:
df.printSchema()

root
 |-- Company Name: string (nullable = true)
 |-- OpenCorporates: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP: string (nullable = true)
 |-- PPP Amount: string (nullable = true)
 |-- Jobs Reported: string (nullable = true)
 |-- jurisdiction code: string (nullable = true)
 |-- Company Number: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Incorporation Date: string (nullable = true)
 |-- Company Type: string (nullable = true)
 |-- Jurisdiction: string (nullable = true)
 |-- Registered Address: string (nullable = true)
 |-- Agent_first: string (nullable = true)
 |-- agent_middle_name: string (nullable = true)
 |-- agent_middle_initial: string (nullable = true)
 |-- agent_last: string (nullable = true)
 |-- Full Name: string (nullable = true)
 |-- Agent Address: string (nullable = true)
 |-- _c20: string (nullable = true)
 |-- _c21: string (nullable = true)



In [60]:
df.select("City", "State", "Status", "agent_last").show(5)

+-----------+-----+------+----------+
|       City|State|Status|agent_last|
+-----------+-----+------+----------+
| Haleyville|   AL|Exists|     HICKS|
|       NULL| NULL|  NULL|      NULL|
|Saint Louis|   MO|Active|Beckemeier|
|     Mobile|   AL|Exists|   SHERMAN|
|       NULL| NULL|  NULL|      NULL|
+-----------+-----+------+----------+
only showing top 5 rows



In [61]:
df.select("City", "State", "Status", "agent_last").show(10)

+-----------+-----+-------------+----------+
|       City|State|       Status|agent_last|
+-----------+-----+-------------+----------+
| Haleyville|   AL|       Exists|     HICKS|
|       NULL| NULL|         NULL|      NULL|
|Saint Louis|   MO|       Active|Beckemeier|
|     Mobile|   AL|       Exists|   SHERMAN|
|       NULL| NULL|         NULL|      NULL|
|   Meredith|   NH|Good Standing|     Brown|
|Osage Beach|   MO|       Active|     SUPER|
|       NULL| NULL|         NULL|      NULL|
|   Columbia|   MO|       Active|    Martin|
|      Foley|   AL|       Exists|    ALLDAY|
+-----------+-----+-------------+----------+
only showing top 10 rows



In [62]:
df.select("City", "State", "Status", "agent_last").limit(5).show()

+-----------+-----+------+----------+
|       City|State|Status|agent_last|
+-----------+-----+------+----------+
| Haleyville|   AL|Exists|     HICKS|
|       NULL| NULL|  NULL|      NULL|
|Saint Louis|   MO|Active|Beckemeier|
|     Mobile|   AL|Exists|   SHERMAN|
|       NULL| NULL|  NULL|      NULL|
+-----------+-----+------+----------+



In [63]:
df.count()

71559

In [64]:
# drop missing values

df = df.na.drop()
df.show()

+--------------------+--------------------+-------+--------------------+-----------+----------+-------------+-----------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+---------------+--------------------+--------------------+------------+--------------------+--------------------+--------------------+--------------------+
|        Company Name|      OpenCorporates|   City|               State|        ZIP|PPP Amount|Jobs Reported|jurisdiction code|Company Number|              Status|  Incorporation Date|        Company Type|        Jurisdiction|  Registered Address|    Agent_first|   agent_middle_name|agent_middle_initial|  agent_last|           Full Name|       Agent Address|                _c20|                _c21|
+--------------------+--------------------+-------+--------------------+-----------+----------+-------------+-----------------+--------------+--------------------+--------------------+----------

In [65]:
df.select("City", "State", "Status", "agent_last").limit(2).show(20)

+-------+--------------------+--------------------+------------+
|   City|               State|              Status|  agent_last|
+-------+--------------------+--------------------+------------+
| LLC"""|http://opencorpor...|               us_nm|           M|
|Tomball|                  TX|        In Existence|Gurrusquieta|
|Wichita|                  KS|Active And In Goo...|          TY|
|  Davis|                  CA|              Active|      LATINI|
+-------+--------------------+--------------------+------------+



## Spark SQL

In [74]:
df = spark.read.csv("FinalBV_V8.csv", header=True, inferSchema=True)

df.show()

+--------------------+--------------------+--------------+-----+-----+----------+-------------+-----------------+--------------+-------------+--------------------+--------------------+------------------+--------------------+-----------+-----------------+--------------------+----------+---------------+--------------------+----+----+
|        Company Name|      OpenCorporates|          City|State|  ZIP|PPP Amount|Jobs Reported|jurisdiction code|Company Number|       Status|  Incorporation Date|        Company Type|      Jurisdiction|  Registered Address|Agent_first|agent_middle_name|agent_middle_initial|agent_last|      Full Name|       Agent Address|_c20|_c21|
+--------------------+--------------------+--------------+-----+-----+----------+-------------+-----------------+--------------+-------------+--------------------+--------------------+------------------+--------------------+-----------+-----------------+--------------------+----------+---------------+--------------------+----+----

In [75]:
# create tempview
df.createOrReplaceTempView("finalbv")

In [76]:
df.printSchema()

root
 |-- Company Name: string (nullable = true)
 |-- OpenCorporates: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP: string (nullable = true)
 |-- PPP Amount: string (nullable = true)
 |-- Jobs Reported: string (nullable = true)
 |-- jurisdiction code: string (nullable = true)
 |-- Company Number: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Incorporation Date: string (nullable = true)
 |-- Company Type: string (nullable = true)
 |-- Jurisdiction: string (nullable = true)
 |-- Registered Address: string (nullable = true)
 |-- Agent_first: string (nullable = true)
 |-- agent_middle_name: string (nullable = true)
 |-- agent_middle_initial: string (nullable = true)
 |-- agent_last: string (nullable = true)
 |-- Full Name: string (nullable = true)
 |-- Agent Address: string (nullable = true)
 |-- _c20: string (nullable = true)
 |-- _c21: string (nullable = true)



In [79]:
df.select("City", "State", "OpenCorporates", "agent_last").show()

+--------------+-----+--------------------+----------+
|          City|State|      OpenCorporates|agent_last|
+--------------+-----+--------------------+----------+
|    Haleyville|   AL|http://opencorpor...|     HICKS|
|          NULL| NULL|           AL 35565"|      NULL|
|   Saint Louis|   MO|http://opencorpor...|Beckemeier|
|        Mobile|   AL|http://opencorpor...|   SHERMAN|
|          NULL| NULL|           AL 36695"|      NULL|
|      Meredith|   NH|http://opencorpor...|     Brown|
|   Osage Beach|   MO|http://opencorpor...|     SUPER|
|          NULL| NULL|           MO 65065"|      NULL|
|      Columbia|   MO|http://opencorpor...|    Martin|
|         Foley|   AL|http://opencorpor...|    ALLDAY|
|          NULL| NULL|           AL 36535"|      NULL|
|   Springfield|   MO|http://opencorpor...|      Hite|
|          NULL| NULL|           MO 65809"|      NULL|
|    Birmingham|   AL|http://opencorpor...|      HALL|
|          NULL| NULL|           AL 35124"|      NULL|
|Webster G

In [85]:
df.select("City", "State", "OpenCorporates", "agent_last") \
.filter("City == 'Columbia'") \
.filter("agent_last == 'Davidson'") \
.show()

+--------+-----+--------------------+----------+
|    City|State|      OpenCorporates|agent_last|
+--------+-----+--------------------+----------+
|Columbia|   MO|http://opencorpor...|  Davidson|
+--------+-----+--------------------+----------+



In [86]:
new_df = spark.sql("SELECT City, State, OpenCorporates, agent_last FROM finalbv WHERE City='Columbia'")

new_df.show()

+--------+-----+--------------------+------------+
|    City|State|      OpenCorporates|  agent_last|
+--------+-----+--------------------+------------+
|Columbia|   MO|http://opencorpor...|      Martin|
|Columbia|   MO|http://opencorpor...|       Jones|
|Columbia|   MO|http://opencorpor...|        Rost|
|Columbia|   MO|http://opencorpor...|      Feaser|
|Columbia|   MO|http://opencorpor...|WELLS-MORGAN|
|Columbia|   MO|http://opencorpor...|      Speake|
|Columbia|   MO|http://opencorpor...|       Coley|
|Columbia|   MO|http://opencorpor...|     Rutiaga|
|Columbia|   MS|http://opencorpor...|      Vander|
|Columbia|   MO|http://opencorpor...|    Davidson|
|Columbia|   MO|http://opencorpor...|        Lutz|
|Columbia|   MO|http://opencorpor...|     Nappier|
|Columbia|   MO|http://opencorpor...|    Patchett|
|Columbia|   MO|http://opencorpor...|        Lowe|
|Columbia|   MO|http://opencorpor...|      Wagner|
|Columbia|   MO|http://opencorpor...|      Dorado|
|Columbia|   MO|http://opencorp

## Output to a csv file

In [87]:
new_df.write.csv("final_bv_columbia", header=True)

In [88]:
new_df.repartition(1).write.mode("overwrite").option("header", "true").csv("final_bv_repartition")

In [89]:
new_df.coalesce(1).write.mode("overwrite").option("header", "true").csv("output")

In [90]:
new_df.collect()

[Row(City='Columbia', State='MO', OpenCorporates='http://opencorporates.com/companies/us_mo/LC0058574', agent_last='Martin'),
 Row(City='Columbia', State='MO', OpenCorporates='http://opencorporates.com/companies/us_mo/LC001537959', agent_last='Jones'),
 Row(City='Columbia', State='MO', OpenCorporates='http://opencorporates.com/companies/us_mo/LC0682219', agent_last='Rost'),
 Row(City='Columbia', State='MO', OpenCorporates='http://opencorporates.com/companies/us_mo/LC001536614', agent_last='Feaser'),
 Row(City='Columbia', State='MO', OpenCorporates='http://opencorporates.com/companies/us_mo/LC001432218', agent_last='WELLS-MORGAN'),
 Row(City='Columbia', State='MO', OpenCorporates='http://opencorporates.com/companies/us_mo/CC0617243', agent_last='Speake'),
 Row(City='Columbia', State='MO', OpenCorporates='http://opencorporates.com/companies/us_mo/LC0879806', agent_last='Coley'),
 Row(City='Columbia', State='MO', OpenCorporates='http://opencorporates.com/companies/us_mo/LC001654935', agen

In [92]:
new_df.take(3)

[Row(City='Columbia', State='MO', OpenCorporates='http://opencorporates.com/companies/us_mo/LC0058574', agent_last='Martin'),
 Row(City='Columbia', State='MO', OpenCorporates='http://opencorporates.com/companies/us_mo/LC001537959', agent_last='Jones'),
 Row(City='Columbia', State='MO', OpenCorporates='http://opencorporates.com/companies/us_mo/LC0682219', agent_last='Rost')]

In [93]:
# save file to parquet

new_df.repartition(1).write.parquet("output/students_parquet")

In [94]:
# read parquet

df = spark.read.parquet("output/students_parquet")

df.show()

+--------+-----+--------------------+------------+
|    City|State|      OpenCorporates|  agent_last|
+--------+-----+--------------------+------------+
|Columbia|   MO|http://opencorpor...|      Martin|
|Columbia|   MO|http://opencorpor...|       Jones|
|Columbia|   MO|http://opencorpor...|        Rost|
|Columbia|   MO|http://opencorpor...|      Feaser|
|Columbia|   MO|http://opencorpor...|WELLS-MORGAN|
|Columbia|   MO|http://opencorpor...|      Speake|
|Columbia|   MO|http://opencorpor...|       Coley|
|Columbia|   MO|http://opencorpor...|     Rutiaga|
|Columbia|   MS|http://opencorpor...|      Vander|
|Columbia|   MO|http://opencorpor...|    Davidson|
|Columbia|   MO|http://opencorpor...|        Lutz|
|Columbia|   MO|http://opencorpor...|     Nappier|
|Columbia|   MO|http://opencorpor...|    Patchett|
|Columbia|   MO|http://opencorpor...|        Lowe|
|Columbia|   MO|http://opencorpor...|      Wagner|
|Columbia|   MO|http://opencorpor...|      Dorado|
|Columbia|   MO|http://opencorp